<a href="https://colab.research.google.com/github/rustamli-nazrin/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rustamli-nazrin/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The purpose of this rule is to identify content that should be refreshed first.               
The rule uses two signals:              
Days since last update (staleness) – older content is more likely to need updating.         
Search impressions (traffic opportunity) – refreshing pages that already receive impressions may produce larger SEO gains.         


Signal check     
Signal 1: Staleness       
Hypothesis: Older pages are more likely to lose performance and benefit from a refresh.        
Verdict: CONFIRMED     
Signal 2: Search Impressions     
Hypothesis:
Pages with higher impressions represent larger optimization opportunities.       
Verdict: CONFIRMED

In [ ]:
import os

print(os.listdir())

['.config', 'content_refresh_anonymized (1).csv', 'sample_data']


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving content_refresh_anonymized (1).csv to content_refresh_anonymized (1) (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized (1).csv")

print("Dataset size:", len(df))

# ---------- Signal 1 ----------
staleness = (
    df.groupby("freshness_tier")
    .agg(
        n=("content_id","count"),
        avg_impressions=("impressions_90d","mean"),
        avg_clicks=("clicks_90d","mean")
    )
)

print("\nSignal 1 - Freshness")
print(staleness)

# ---------- Signal 2 ----------
traffic = (
    df.groupby("impression_tier")
    .agg(
        n=("content_id","count"),
        avg_age=("days_since_last_update","mean")
    )
)

print("\nSignal 2 - Impression Tier")
print(traffic)

Dataset size: 30000

Signal 1 - Freshness
                    n  avg_impressions  avg_clicks
freshness_tier                                    
0-30            20480      4199.614062   13.727393
181+              174      1172.448276    2.672414
31-90             175      6506.748571    9.685714
91-180           9171      7486.665140   21.766765

Signal 2 - Impression Tier
                     n    avg_age
impression_tier                  
excellent         1078  58.338590
good              7205  55.259681
low              11248  36.057344
moderate         10469  49.320948


## 2. Build the ranked queue (writes the CSV)

Baseline Scoring Rule       
A page receives a higher score when it is both:
* old
* receiving meaningful search impressions  

Older pages with higher visibility should be refreshed first.
      
Action Label:
REFRESH_CONTENT

In [ ]:
import os

# Normalize values
age_score = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
)

traffic_score = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)

df["baseline_score"] = (
    age_score * 0.6 +
    traffic_score * 0.4
)

def reason(row):
    if row["impressions_90d"] >= df["impressions_90d"].median():
        return "STALE_HIGH_TRAFFIC"
    elif row["days_since_last_update"] >= df["days_since_last_update"].median():
        return "STALE_MEDIUM_TRAFFIC"
    else:
        return "LOW_PRIORITY"

df["reason_code"] = df.apply(reason, axis=1)

df["action"] = "REFRESH_CONTENT"

ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(ranked[
    ["content_id",
     "baseline_score",
     "reason_code",
     "action"]
].head(10))

                 content_id  baseline_score           reason_code  \
26242  content_55a5b1c46474        0.600027  STALE_MEDIUM_TRAFFIC   
29384  content_f6fdf87348f6        0.600002  STALE_MEDIUM_TRAFFIC   
4606   content_3f3576c295f5        0.600001  STALE_MEDIUM_TRAFFIC   
24216  content_1b4ec72dafd4        0.598393  STALE_MEDIUM_TRAFFIC   
18440  content_8d56efff1e71        0.598392  STALE_MEDIUM_TRAFFIC   
6653   content_5fe46e04994d        0.567292    STALE_HIGH_TRAFFIC   
6962   content_f01216059a6a        0.538914  STALE_MEDIUM_TRAFFIC   
8631   content_e2b702f4f92b        0.537289  STALE_MEDIUM_TRAFFIC   
15608  content_06e19c6486b0        0.537273  STALE_MEDIUM_TRAFFIC   
29400  content_2dba2b1f9536        0.509901    STALE_HIGH_TRAFFIC   

                action  
26242  REFRESH_CONTENT  
29384  REFRESH_CONTENT  
4606   REFRESH_CONTENT  
24216  REFRESH_CONTENT  
18440  REFRESH_CONTENT  
6653   REFRESH_CONTENT  
6962   REFRESH_CONTENT  
8631   REFRESH_CONTENT  
15608  REFRESH_

##3. Top 10 review


Rank,
Action,
Why it's there,
What would make it wrong?


1. REFRESH_CONTENT   
* Very old content with a high baseline score (STALE_MEDIUM_TRAFFIC).
* The page may have been updated recently but the dataset has not yet reflected the change.           
2. REFRESH_CONTENT
* Old content prioritized by the baseline scoring rule.
* Traffic may be seasonal rather than caused by outdated content.
3. REFRESH_CONTENT
* High score due to content age.
* The topic may already be evergreen and not require refreshing.
4. REFRESH_CONTENT
* Ranked highly because of staleness.
* Business value may be low despite the score.
5. REFRESH_CONTENT
* Old content selected for refresh.
* Performance could recover without intervention.
6. REFRESH_CONTENT
* STALE_HIGH_TRAFFIC indicates both staleness and strong traffic opportunity.
* High impressions do not always mean the content needs updating.
7. REFRESH_CONTENT
* Selected because of content age.
* The page could already be scheduled for a refresh.
8. REFRESH_CONTENT
* High baseline score based on the rule.
* The observed traffic pattern could be temporary.
9. REFRESH_CONTENT
* High traffic combined with staleness.
* External factors rather than content quality may explain performance.
10. REFRESH_CONTENT
* STALE_HIGH_TRAFFIC makes it a good refresh candidate.
* The page may already satisfy user intent and not benefit from changes.

## 4. Weak picks + leakage check

Weak Picks     
Some lower-ranked pages may not truly require refreshing because: traffic changes can be seasonal; older content is not always outdated; some pages have low business value despite receiving impressions.


Leakage Check   
The baseline score only uses observable current features:
days since last update
impressions    
No future information, product flags, labels, or target variables were used when calculating the score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.